# 🧠 Step 4 — RGB 데이터셋 YOLOv8 전용 학습 (최종본)
## 14,000장 RGB 데이터 기반 최적화 학습 + MLflow 실시간 모니터링

### ✅ 학습 전략
| 항목 | 설정 내용 |
|------|-----------|
| **데이터** | RGB 이미지 (Train 14,000 / Val 2,000) |
| **모델** | `yolov8m.pt` (Medium - 정확도와 속도 균형) |
| **배치 사이즈** | 16 (RTX 4070 8GB VRAM 최적화) |
| **조기 종료** | `patience=15` (성능 개선 없을 시 자동 정지) |
| **실시간 모니터링** | MLflow 연동 (mAP, Loss 그래프 확인) |

In [ ]:
# ── 0. 환경 설정 ─────────────────────────────────────────────
import os, gc, time, random
from pathlib import Path

import torch
import mlflow
import mlflow.pytorch
from ultralytics import YOLO

# CUDA 메모리 효율화
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'MLflow   : {mlflow.__version__}')

In [ ]:
# ── 1. 경로 및 하이퍼파라미터 설정 ─────────────────────────────
BASE_DIR   = Path(r'C:\Users\SSAFY\Desktop\ai-teach')
YOLO_DIR   = BASE_DIR / 'yolo_dataset'
RUNS_DIR   = BASE_DIR / 'runs'
MLFLOW_URI = BASE_DIR / 'mlruns'

CFG = {
    'model'       : 'yolov8m.pt',
    'data'        : (YOLO_DIR / 'data.yaml').as_posix(),
    'project'     : RUNS_DIR.as_posix(),
    'name'        : 'ALS_RGB_Training_Final',
    'epochs'      : 100,
    'batch'       : 16,
    'imgsz'       : 640,
    'patience'    : 15,
    'device'      : 0,
    'workers'     : 4,
    'optimizer'   : 'AdamW',
    'lr0'         : 0.001,
    'amp'         : True,
    'save_period' : 10,
}

In [ ]:
# ── 2. MLflow 실험 및 URI 최적화 ─────────────────────────────────
mlflow_path = MLFLOW_URI.as_posix()
mlflow.set_tracking_uri(f'file:///{mlflow_path}')

experiment_name = 'ALS_EyeTracking_RGB_Alpha'
mlflow.set_experiment(experiment_name)

print(f'📍 실험 기록 저장소: {mlflow.get_tracking_uri()}')
print(f'🧪 현재 실험 이름: {experiment_name}')
print()
print('📊 MLflow UI로 실시간 그래프를 보는 방법 (별도 터미널):')
print(f'   mlflow ui --backend-store-uri file:///{mlflow_path} --port 5000')

In [ ]:
# ── 3. 메모리 정리 및 체크포인트 확인 ───────────────────────────
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM 상태: {free/1e9:.2f} GB / {total/1e9:.2f} GB 여유')

last_weight = RUNS_DIR / CFG['name'] / 'weights' / 'last.pt'
RESUME = last_weight.exists()
print(f'RESUME 모드: {"활성화 (기존 학습 이어서 하기)" if RESUME else "비활성화 (새로운 학습 시작)"}')

In [ ]:
# ── 4. MLflow Callback 정의 ────────────────────────────────────
class MLflowCallback:
    def __init__(self, run):
        self.run = run

    def on_train_epoch_end(self, trainer):
        metrics = trainer.metrics
        epoch   = trainer.epoch
        logs = {
            'train/box_loss' : float(trainer.loss_items[0]) if hasattr(trainer, 'loss_items') else 0,
            'train/cls_loss' : float(trainer.loss_items[1]) if hasattr(trainer, 'loss_items') else 0,
            'train/dfl_loss' : float(trainer.loss_items[2]) if hasattr(trainer, 'loss_items') else 0,
        }
        if metrics:
            logs.update({
                'val/mAP50'    : float(metrics.get('metrics/mAP50(B)', 0)),
                'val/mAP50-95' : float(metrics.get('metrics/mAP50-95(B)', 0)),
                'val/precision': float(metrics.get('metrics/precision(B)', 0)),
                'val/recall'   : float(metrics.get('metrics/recall(B)', 0)),
            })
        mlflow.log_metrics(logs, step=epoch)

    def on_fit_epoch_end(self, trainer):
        lrs = {f'lr/pg{i}': lr for i, lr in enumerate(trainer.scheduler.get_last_lr())}
        mlflow.log_metrics(lrs, step=trainer.epoch)

print('MLflow 로깅 엔진 장착 완료! ✅')

In [ ]:
# ── 5. 학습 가동 🚀 ───────────────────────────────────────────
run_name = f'yolov8m_rgb_{time.strftime("%m%d_%H%M")}'

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params(CFG)
    
    # 모델 로드
    model_path = str(last_weight) if RESUME else CFG['model']
    model = YOLO(model_path)
    
    # 콜백 등록
    cb = MLflowCallback(run)
    model.add_callback('on_train_epoch_end', cb.on_train_epoch_end)
    model.add_callback('on_fit_epoch_end',   cb.on_fit_epoch_end)
    
    # 학습 파라미터 구성
    train_args = {k: v for k, v in CFG.items() if k != 'model'}
    if RESUME:
        train_args['resume'] = True
    
    print(f'🔥 학습 시작! (Run: {run_name})')
    results = model.train(**train_args)
    
    print(f'🎉 학습 완료!')
    print(f'결과물 위치: {results.save_dir}')

In [ ]:
# ── 6. 모델 검증 및 추론 테스트 (학습 완료 후 실행) ──────────────
import random

# 실제 가중치 경로 (필요시 Final3 등으로 수정)
best_weight = RUNS_DIR / CFG['name'] / 'weights' / 'best.pt'
if not best_weight.exists():
    # Final3 폴더 등이 있는지 유연하게 체크
    alt_path = RUNS_DIR / (CFG['name'] + '3') / 'weights' / 'best.pt'
    if alt_path.exists(): best_weight = alt_path

if best_weight.exists():
    print(f'✅ 최적 가중치 로드: {best_weight}')
    model = YOLO(str(best_weight))
    
    # 1. 검증 데이터셋 수치 평가
    print('\n📊 검증 데이터셋 점수 측정 중...')
    val_results = model.val(data=CFG['data'], imgsz=CFG['imgsz'])
    print(f'-- 최종 mAP50    : {val_results.box.map50:.4f}')
    print(f'-- 최종 mAP50-95 : {val_results.box.map:.4f}')
    
    # 2. 샘플 이미지 추론 테스트 (눈으로 확인)
    print('\n🖥️ 랜덤 샘플 이미지 추론 테스트 중...')
    val_img_dir = YOLO_DIR / 'images' / 'val'
    all_val_images = list(val_img_dir.glob('*.jpg'))
    
    if all_val_images:
        # 매번 다른 결과를 위해 무작위 3장 선택
        sample_images = random.sample(all_val_images, min(3, len(all_val_images)))
        inference_results = model.predict(source=[str(p) for p in sample_images], save=True, imgsz=CFG['imgsz'])
        print(f'✅ 랜덤 샘플 추론 완료! 결과 폴더: {inference_results[0].save_dir}')
else:
    print('❌ 안내: best.pt 파일을 찾을 수 없습니다. 학습이 완료되었는지 확인해주세요.')